# 06 — Contexte météorologique et bonus VAE

Mode utilisé dans ce support : **smoke**, destiné à vérifier la chaîne complète. Les valeurs commentées proviennent du rapport calculé, jamais de scores inventés.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
REPORT = ROOT / "outputs" / "reports" / "results_smoke.json"
if not REPORT.exists():
    raise FileNotFoundError("Exécutez d'abord : .venv\\Scripts\\python scripts\\run_smoke.py --mode smoke")
results = json.loads(REPORT.read_text(encoding="utf-8"))
sns.set_theme(style="whitegrid")

## Auto-encodeur conditionnel

Nous calculons $z=f(x)$ puis $\hat x=g(z,c)$ avec $c$ = température moyenne et calendrier sin/cos. La comparaison utilise exactement les mêmes fenêtres. L'architecture ne garantit pas que $z$ soit indépendant de la météo ; changer la température à latent fixé n'est pas une intervention causale.

In [ ]:
baseline=results['daily'][f"nonlinear_ae_{min(map(int,[k.split('_')[-1] for k in results['daily'] if k.startswith('nonlinear_ae_')]))}"]
context=results['context']
display(pd.DataFrame({'sans contexte':baseline,'conditionnel':context}).T[['rmse_observed','mae_observed']])
pd.Series(context['rmse_by_temperature_tercile']).plot.bar(figsize=(8,3)); plt.ylabel('RMSE moyenne'); plt.title('Erreur conditionnelle par tercile de température'); plt.xticks(rotation=20,ha='right'); plt.show()

## Bonus VAE

Le VAE produit $\mu(x)$ et $\log\sigma^2(x)$, puis

$$z=\mu+\sigma\odot\epsilon,\qquad \epsilon\sim\mathcal N(0,I),$$

et minimise reconstruction + $\beta D_{KL}(q(z|x)\|\mathcal N(0,I))$. La régularisation facilite interpolations et génération mais peut dégrader la fidélité. Elle ne garantit pas de meilleurs clusters ni de meilleures anomalies.

In [ ]:
pd.DataFrame({'AE':baseline,'VAE':results['vae_bonus']}).T[['rmse_observed','mae_observed']]

## Limites et questions

Le smoke test n'a que deux époques et une fenêtre climatique courte ; l'erreur par tranche de température et les épisodes froids doivent être étudiés en mode full. Questions : comment vérifier empiriquement si le latent contient encore la température ? Que signifie interpoler entre deux foyers ? Quel rôle joue $eta$ dans le compromis netteté/régularité ?